In [1]:
import json

with open(
    "../data/processed/train_papers.json",
    "r",
    encoding="utf-8"
) as f:

    train_papers = json.load(f)


with open(
    "../data/processed/validation_papers.json",
    "r",
    encoding="utf-8"
) as f:

    validation_papers = json.load(f)


with open(
    "../data/processed/test_papers.json",
    "r",
    encoding="utf-8"
) as f:

    test_papers = json.load(f)


print("Train:", len(train_papers))
print("Validation:", len(validation_papers))
print("Test:", len(test_papers))

Train: 124
Validation: 26
Test: 29


title + abstract

In [2]:
def combine_text(paper):

    title = paper["title"]
    abstract = paper["abstract"]

    return title + " " + abstract

In [3]:
example = combine_text(train_papers[0])

print(example)

End-to-End Speaker Diarization as Post-Processing This paper investigates the utilization of an end-to-end diarization model as post-processing of conventional clustering-based diarization. Clustering-based diarization methods partition frames into clusters of the number of speakers; thus, they typically cannot handle overlapping speech because each frame is assigned to one speaker. On the other hand, some end-to-end diarization methods can handle overlapping speech by treating the problem as multi-label classification. Although some methods can treat a flexible number of speakers, they do not perform well when the number of speakers is large. To compensate for each other's weakness, we propose to use a two-speaker end-to-end diarization method as post-processing of the results obtained by a clustering-based method. We iteratively select two speakers from the results and update the results of the two speakers to improve the overlapped region. Experimental results show that the proposed

Simple Clean

In [4]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [5]:
for paper in train_papers:

    text = combine_text(paper)

    cleaned = clean_text(text)

    paper["clean_text"] = cleaned

In [6]:
for paper in validation_papers:

    text = combine_text(paper)

    paper["clean_text"] = clean_text(text)


for paper in test_papers:

    text = combine_text(paper)

    paper["clean_text"] = clean_text(text)

In [7]:
print("TITLE:")
print(train_papers[0]["title"])

print("\nABSTRACT:")
print(train_papers[0]["abstract"])

print("\nCLEAN TEXT:")
print(train_papers[0]["clean_text"])

TITLE:
End-to-End Speaker Diarization as Post-Processing

ABSTRACT:
This paper investigates the utilization of an end-to-end diarization model as post-processing of conventional clustering-based diarization. Clustering-based diarization methods partition frames into clusters of the number of speakers; thus, they typically cannot handle overlapping speech because each frame is assigned to one speaker. On the other hand, some end-to-end diarization methods can handle overlapping speech by treating the problem as multi-label classification. Although some methods can treat a flexible number of speakers, they do not perform well when the number of speakers is large. To compensate for each other's weakness, we propose to use a two-speaker end-to-end diarization method as post-processing of the results obtained by a clustering-based method. We iteratively select two speakers from the results and update the results of the two speakers to improve the overlapped region. Experimental results show

TF-IDF

In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

print("Stopword count:", len(stopwords))

Stopword count: 318


In [9]:
def remove_stopwords(text):
    words = text.split()

    words = [
        word
        for word in words
        if word not in stopwords
    ]

    return " ".join(words)

In [10]:
example_clean = train_papers[0]["clean_text"]

example_no_stopwords = remove_stopwords(example_clean)

print("BEFORE:")
print(example_clean)

print("\n" + "=" * 60)

print("AFTER STOPWORD REMOVAL:")
print(example_no_stopwords)

BEFORE:
end to end speaker diarization as post processing this paper investigates the utilization of an end to end diarization model as post processing of conventional clustering based diarization clustering based diarization methods partition frames into clusters of the number of speakers thus they typically cannot handle overlapping speech because each frame is assigned to one speaker on the other hand some end to end diarization methods can handle overlapping speech by treating the problem as multi label classification although some methods can treat a flexible number of speakers they do not perform well when the number of speakers is large to compensate for each other s weakness we propose to use a two speaker end to end diarization method as post processing of the results obtained by a clustering based method we iteratively select two speakers from the results and update the results of the two speakers to improve the overlapped region experimental results show that the proposed al

In [11]:
for paper in train_papers:
    paper["clean_text_no_stopwords"] = remove_stopwords(
        paper["clean_text"]
    )

for paper in validation_papers:
    paper["clean_text_no_stopwords"] = remove_stopwords(
        paper["clean_text"]
    )

for paper in test_papers:
    paper["clean_text_no_stopwords"] = remove_stopwords(
        paper["clean_text"]
    )

In [12]:
print("CLEAN TEXT:")
print(train_papers[0]["clean_text"])

print("\n" + "=" * 60)

print("WITHOUT STOPWORDS:")
print(train_papers[0]["clean_text_no_stopwords"])

CLEAN TEXT:
end to end speaker diarization as post processing this paper investigates the utilization of an end to end diarization model as post processing of conventional clustering based diarization clustering based diarization methods partition frames into clusters of the number of speakers thus they typically cannot handle overlapping speech because each frame is assigned to one speaker on the other hand some end to end diarization methods can handle overlapping speech by treating the problem as multi label classification although some methods can treat a flexible number of speakers they do not perform well when the number of speakers is large to compensate for each other s weakness we propose to use a two speaker end to end diarization method as post processing of the results obtained by a clustering based method we iteratively select two speakers from the results and update the results of the two speakers to improve the overlapped region experimental results show that the propose

Normal TF-IDF

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=False
)

X_train = vectorizer.fit_transform(
    [paper["clean_text"] for paper in train_papers]
)

X_validation = vectorizer.transform(
    [paper["clean_text"] for paper in validation_papers]
)

X_test = vectorizer.transform(
    [paper["clean_text"] for paper in test_papers]
)

NO Stopwords  TF-IDF

In [14]:
vectorizer_sw = TfidfVectorizer(
    lowercase=False
)

X_train_sw = vectorizer_sw.fit_transform(
    [paper["clean_text_no_stopwords"] for paper in train_papers]
)

X_validation_sw = vectorizer_sw.transform(
    [paper["clean_text_no_stopwords"] for paper in validation_papers]
)

X_test_sw = vectorizer_sw.transform(
    [paper["clean_text_no_stopwords"] for paper in test_papers]
)

In [15]:
print("NORMAL TF-IDF")
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

print("\nSTOPWORD REMOVED TF-IDF")
print("Train:", X_train_sw.shape)
print("Validation:", X_validation_sw.shape)
print("Test:", X_test_sw.shape)

NORMAL TF-IDF
Train: (124, 3729)
Validation: (26, 3729)
Test: (29, 3729)

STOPWORD REMOVED TF-IDF
Train: (124, 3522)
Validation: (26, 3522)
Test: (29, 3522)


Training

In [19]:
import json

with open(
    "../data/processed/papers_merged.json",
    "r",
    encoding="utf-8"
) as f:

    papers = json.load(f)

print("Total papers:", len(papers))

Total papers: 179


In [20]:
import numpy as np

label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

print(label_names)

['NLP', 'Computer Vision', 'Machine Learning', 'Robotics']


In [21]:
Y = np.zeros(
    (len(papers), len(label_names)),
    dtype=int
)

for i, paper in enumerate(papers):

    for label in paper["labels"]:

        label_index = label_names.index(label)

        Y[i, label_index] = 1

In [22]:
print("Y shape:", Y.shape)

Y shape: (179, 4)


In [23]:
for i, label in enumerate(label_names):

    print(
        label,
        ":",
        Y[:, i].sum()
    )

NLP : 50
Computer Vision : 50
Machine Learning : 50
Robotics : 50


In [24]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

X = np.arange(len(papers))

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    msss.split(X, Y)
)

Y_temp = Y[temp_idx]

X_temp = np.arange(len(temp_idx))

msss_temp = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_relative_idx, test_relative_idx = next(
    msss_temp.split(X_temp, Y_temp)
)

val_idx = temp_idx[val_relative_idx]
test_idx = temp_idx[test_relative_idx]

In [25]:
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 124
Validation: 26
Test: 29


In [26]:
Y_train = Y[train_idx]
Y_validation = Y[val_idx]
Y_test = Y[test_idx]

print("Y_train:", Y_train.shape)
print("Y_validation:", Y_validation.shape)
print("Y_test:", Y_test.shape)

Y_train: (124, 4)
Y_validation: (26, 4)
Y_test: (29, 4)


## One-vs-Rest Logistic Regression

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report


In [28]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    )
)

In [29]:
model.fit(
    X_train,
    Y_train
)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](4,)","[0,1,2,3]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42)]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,4
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,3729
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42


In [30]:
Y_val_pred = model.predict(
    X_validation
)

In [31]:
print("Prediction shape:", Y_val_pred.shape)

Prediction shape: (26, 4)


In [32]:
print("TRUE LABELS:")
print(Y_validation[:5])

print("\nPREDICTED LABELS:")
print(Y_val_pred[:5])

TRUE LABELS:
[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]

PREDICTED LABELS:
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]


In [35]:
from sklearn.metrics import f1_score, classification_report

micro_f1 = f1_score(
    Y_validation,
    Y_val_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    Y_validation,
    Y_val_pred,
    average="macro",
    zero_division=0
)

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)

Micro F1: 0.0
Macro F1: 0.0


In [36]:
print("\nClassification Report:\n")

print(
    classification_report(
        Y_validation,
        Y_val_pred,
        target_names=label_names,
        zero_division=0
    )
)


Classification Report:

                  precision    recall  f1-score   support

             NLP       0.00      0.00      0.00         7
 Computer Vision       0.00      0.00      0.00         7
Machine Learning       0.00      0.00      0.00         7
        Robotics       0.00      0.00      0.00         7

       micro avg       0.00      0.00      0.00        28
       macro avg       0.00      0.00      0.00        28
    weighted avg       0.00      0.00      0.00        28
     samples avg       0.00      0.00      0.00        28



In [37]:
print("Toplam pozitif tahmin:", Y_val_pred.sum())
print("Toplam gerçek pozitif:", Y_validation.sum())

print("\nTahmin edilen label sayıları:")
for i, label in enumerate(label_names):
    print(label, ":", Y_val_pred[:, i].sum())

Toplam pozitif tahmin: 0
Toplam gerçek pozitif: 28

Tahmin edilen label sayıları:
NLP : 0
Computer Vision : 0
Machine Learning : 0
Robotics : 0


In [38]:
decision_scores = model.decision_function(X_validation)

print("Decision scores - ilk 5:")
print(decision_scores[:5])

Decision scores - ilk 5:
[[-0.77486161 -1.06764645 -1.02486012 -1.08414858]
 [-0.94592787 -1.09393262 -0.98496362 -0.94598028]
 [-0.67592074 -1.09144549 -1.06726334 -1.12343744]
 [-0.53776451 -1.14389751 -1.11800346 -1.22550976]
 [-0.42184003 -1.14889034 -0.98059577 -1.19911921]]


In [39]:
print("\nNLP:")
print(decision_scores[:, 0])

print("\nComputer Vision:")
print(decision_scores[:, 1])

print("\nMachine Learning:")
print(decision_scores[:, 2])

print("\nRobotics:")
print(decision_scores[:, 3])


NLP:
[-0.77486161 -0.94592787 -0.67592074 -0.53776451 -0.42184003 -0.23363173
 -1.00857745 -1.06379769 -0.72143115 -1.20640679 -1.15381948 -0.9606086
 -0.87331357 -1.12874687 -1.08128207 -0.9406547  -1.02084659 -0.80491158
 -1.17218796 -1.02385368 -1.07434604 -0.97096784 -1.23901222 -1.18552557
 -1.20880445 -1.23675132]

Computer Vision:
[-1.06764645 -1.09393262 -1.09144549 -1.14389751 -1.14889034 -1.21969483
 -0.90468219 -0.94426575 -0.86180871 -0.81323179 -0.51972031 -0.92945627
 -0.47673413 -0.95438477 -0.79454589 -1.00285931 -1.08851439 -0.84408822
 -0.86919651 -0.92224056 -1.17239859 -1.07800443 -1.22927432 -0.76632499
 -1.1467182  -1.25403454]

Machine Learning:
[-1.02486012 -0.98496362 -1.06726334 -1.11800346 -0.98059577 -1.12395149
 -0.909809   -0.8299172  -0.97295215 -1.06690345 -1.0446795  -0.93904978
 -0.90767063 -0.82159644 -0.78039479 -0.65609399 -0.95229916 -1.06789506
 -0.63016343 -1.04237126 -0.87415564 -0.99186169 -0.98267598 -1.04440224
 -0.91725148 -1.01494136]

Rob

In [40]:
Y_train_pred = model.predict(X_train)

train_micro_f1 = f1_score(
    Y_train,
    Y_train_pred,
    average="micro",
    zero_division=0
)

print("Training Micro F1:", train_micro_f1)
print("Training positive predictions:", Y_train_pred.sum())
print("Training actual positives:", Y_train.sum())

Training Micro F1: 0.0821917808219178
Training positive predictions: 6
Training actual positives: 140


In [41]:
print("TRAIN PAPER:")
print(train_papers[0]["title"])

print("\nTRAIN LABEL:")
print(train_papers[0]["labels"])

print("\nY_train[0]:")
print(Y_train[0])

TRAIN PAPER:
End-to-End Speaker Diarization as Post-Processing

TRAIN LABEL:
['NLP']

Y_train[0]:
[1 0 0 0]


In [42]:
for i in range(10):

    print(
        i,
        "|",
        train_papers[i]["title"],
        "|",
        train_papers[i]["labels"],
        "|",
        Y_train[i]
    )

0 | End-to-End Speaker Diarization as Post-Processing | ['NLP'] | [1 0 0 0]
1 | Regularized Attentive Capsule Network for Overlapped Relation Extraction | ['NLP'] | [1 0 0 0]
2 | Should I visit this place? Inclusion and Exclusion Phrase Mining from Reviews | ['NLP'] | [1 0 0 0]
3 | Speech Synthesis as Augmentation for Low-Resource ASR | ['NLP'] | [1 0 0 0]
4 | I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling | ['NLP', 'Machine Learning'] | [1 0 1 0]
5 | Mitigating the Impact of Speech Recognition Errors on Spoken Question Answering by Adversarial Domain Adaptation | ['NLP'] | [1 0 0 0]
6 | Posterior-regularized REINFORCE for Instance Selection in Distant Supervision | ['NLP', 'Machine Learning'] | [1 0 1 0]
7 | End-to-End Speech Translation with Knowledge Distillation | ['NLP'] | [1 0 0 0]
8 | FAQ Retrieval using Query-Question Similarity and BERT-Based Query-Answer Relevance | ['NLP'] | [1 0 0 0]
9 | Automatic Inference of Minimalist Grammars using an S

In [43]:
print("TRAIN PREDICTION DISTRIBUTION")
print("=" * 50)

for i, label in enumerate(label_names):
    print(
        label,
        "| Actual:",
        Y_train[:, i].sum(),
        "| Predicted:",
        Y_train_pred[:, i].sum()
    )

TRAIN PREDICTION DISTRIBUTION
NLP | Actual: 35 | Predicted: 3
Computer Vision | Actual: 35 | Predicted: 2
Machine Learning | Actual: 35 | Predicted: 0
Robotics | Actual: 35 | Predicted: 1


In [44]:
train_probabilities = model.predict_proba(X_train)

print("Probability shape:", train_probabilities.shape)
print(train_probabilities[:5])

Probability shape: (124, 4)
[[0.41161109 0.24976761 0.19977664 0.22786571]
 [0.46439153 0.22138836 0.24184257 0.19230335]
 [0.42486778 0.21200338 0.25646739 0.21412026]
 [0.48714935 0.20140849 0.2152932  0.19554608]
 [0.43349748 0.25386321 0.37652201 0.2113518 ]]


In [45]:
val_probabilities = model.predict_proba(X_validation)

print(val_probabilities[:5])

[[0.31542838 0.25585092 0.26408179 0.25272174]
 [0.2797045  0.25087846 0.271908   0.27969394]
 [0.33717236 0.25134618 0.25592387 0.24537423]
 [0.36870777 0.24160549 0.24638181 0.22696829]
 [0.39607653 0.24069182 0.27277359 0.23163194]]


In [46]:
print("TRAIN PREDICTION DISTRIBUTION")
print("=" * 50)

for i, label in enumerate(label_names):
    print(
        label,
        "| Actual:",
        Y_train[:, i].sum(),
        "| Predicted:",
        Y_train_pred[:, i].sum()
    )

TRAIN PREDICTION DISTRIBUTION
NLP | Actual: 35 | Predicted: 3
Computer Vision | Actual: 35 | Predicted: 2
Machine Learning | Actual: 35 | Predicted: 0
Robotics | Actual: 35 | Predicted: 1


In [47]:
threshold = 0.30

Y_val_pred_030 = (
    val_probabilities >= threshold
).astype(int)

print("Predicted positives:", Y_val_pred_030.sum())
print("Actual positives:", Y_validation.sum())

Predicted positives: 23
Actual positives: 28


In [48]:
micro_f1_030 = f1_score(
    Y_validation,
    Y_val_pred_030,
    average="micro",
    zero_division=0
)

macro_f1_030 = f1_score(
    Y_validation,
    Y_val_pred_030,
    average="macro",
    zero_division=0
)

print("Threshold:", threshold)
print("Micro F1:", micro_f1_030)
print("Macro F1:", macro_f1_030)

Threshold: 0.3
Micro F1: 0.6666666666666666
Macro F1: 0.668956043956044


In [49]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

for threshold in thresholds:

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    micro = f1_score(
        Y_validation,
        predictions,
        average="micro",
        zero_division=0
    )

    macro = f1_score(
        Y_validation,
        predictions,
        average="macro",
        zero_division=0
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Micro F1: {micro:.3f} | "
        f"Macro F1: {macro:.3f} | "
        f"Predicted Positives: {predictions.sum()}"
    )

Threshold: 0.20 | Micro F1: 0.431 | Macro F1: 0.431 | Predicted Positives: 102
Threshold: 0.25 | Micro F1: 0.533 | Macro F1: 0.540 | Predicted Positives: 77
Threshold: 0.30 | Micro F1: 0.667 | Macro F1: 0.669 | Predicted Positives: 23
Threshold: 0.35 | Micro F1: 0.444 | Macro F1: 0.411 | Predicted Positives: 8
Threshold: 0.40 | Micro F1: 0.069 | Macro F1: 0.062 | Predicted Positives: 1
Threshold: 0.45 | Micro F1: 0.000 | Macro F1: 0.000 | Predicted Positives: 0
Threshold: 0.50 | Micro F1: 0.000 | Macro F1: 0.000 | Predicted Positives: 0


In [50]:
best_threshold = 0.30

Y_val_pred_best = (
    val_probabilities >= best_threshold
).astype(int)

print(
    classification_report(
        Y_validation,
        Y_val_pred_best,
        target_names=label_names,
        zero_division=0
    )
)

                  precision    recall  f1-score   support

             NLP       0.71      0.71      0.71         7
 Computer Vision       0.50      0.43      0.46         7
Machine Learning       0.80      0.57      0.67         7
        Robotics       1.00      0.71      0.83         7

       micro avg       0.74      0.61      0.67        28
       macro avg       0.75      0.61      0.67        28
    weighted avg       0.75      0.61      0.67        28
     samples avg       0.63      0.63      0.63        28



In [51]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

C_values = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]

results = []

for C in C_values:

    model_c = OneVsRestClassifier(
        LogisticRegression(
            C=C,
            max_iter=2000,
            random_state=42
        )
    )

    model_c.fit(X_train, Y_train)

    val_probs = model_c.predict_proba(X_validation)

    val_pred = (
        val_probs >= 0.30
    ).astype(int)

    micro = f1_score(
        Y_validation,
        val_pred,
        average="micro",
        zero_division=0
    )

    macro = f1_score(
        Y_validation,
        val_pred,
        average="macro",
        zero_division=0
    )

    results.append(
        {
            "C": C,
            "Micro F1": micro,
            "Macro F1": macro
        }
    )

    print(
        f"C={C:<5} | "
        f"Micro F1={micro:.3f} | "
        f"Macro F1={macro:.3f}"
    )

C=0.01  | Micro F1=0.000 | Macro F1=0.000
C=0.05  | Micro F1=0.000 | Macro F1=0.000
C=0.1   | Micro F1=0.069 | Macro F1=0.062
C=0.5   | Micro F1=0.622 | Macro F1=0.615
C=1     | Micro F1=0.667 | Macro F1=0.669
C=2     | Micro F1=0.667 | Macro F1=0.669
C=5     | Micro F1=0.667 | Macro F1=0.669
C=10    | Micro F1=0.640 | Macro F1=0.642


In [52]:
C_values = [0.5, 1, 2, 5, 10]
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]

results = []

for C in C_values:

    model_c = OneVsRestClassifier(
        LogisticRegression(
            C=C,
            max_iter=2000,
            random_state=42
        )
    )

    model_c.fit(X_train, Y_train)

    val_probs = model_c.predict_proba(X_validation)

    for threshold in thresholds:

        val_pred = (
            val_probs >= threshold
        ).astype(int)

        micro = f1_score(
            Y_validation,
            val_pred,
            average="micro",
            zero_division=0
        )

        macro = f1_score(
            Y_validation,
            val_pred,
            average="macro",
            zero_division=0
        )

        results.append({
            "C": C,
            "Threshold": threshold,
            "Micro F1": micro,
            "Macro F1": macro
        })

In [57]:
import pandas as pd
results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        "Micro F1",
        ascending=False
    ).to_string(index=False)
)

   C  Threshold  Micro F1  Macro F1
 2.0       0.30  0.666667  0.668956
 1.0       0.30  0.666667  0.668956
 5.0       0.30  0.666667  0.668956
10.0       0.25  0.655172  0.660714
10.0       0.20  0.641026  0.642419
10.0       0.30  0.640000  0.642441
 5.0       0.25  0.637681  0.639512
 2.0       0.25  0.626506  0.630435
 0.5       0.30  0.622222  0.615035
10.0       0.35  0.604651  0.598485
 5.0       0.35  0.604651  0.598485
 2.0       0.35  0.600000  0.585859
 5.0       0.20  0.584270  0.591142
 5.0       0.40  0.564103  0.554040
10.0       0.40  0.564103  0.554040
 1.0       0.25  0.533333  0.539693
 2.0       0.20  0.478632  0.483417
 0.5       0.25  0.444444  0.445005
 1.0       0.35  0.444444  0.411111
 1.0       0.20  0.430769  0.431085
 0.5       0.20  0.424242  0.424242
 2.0       0.40  0.400000  0.372222
 1.0       0.40  0.068966  0.062500
 0.5       0.35  0.068966  0.062500
 0.5       0.40  0.000000  0.000000


In [58]:
train_val_papers = train_papers + validation_papers

print("Train:", len(train_papers))
print("Validation:", len(validation_papers))
print("Train + Validation:", len(train_val_papers))
print("Test:", len(test_papers))

Train: 124
Validation: 26
Train + Validation: 150
Test: 29


test normal text not non stop words

In [59]:
normal_train_texts = [
    paper["abstract"]
    for paper in train_papers
]

normal_validation_texts = [
    paper["abstract"]
    for paper in validation_papers
]

print("Train texts:", len(normal_train_texts))
print("Validation texts:", len(normal_validation_texts))

Train texts: 124
Validation texts: 26


In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

normal_vectorizer = TfidfVectorizer()

X_train_normal = normal_vectorizer.fit_transform(
    normal_train_texts
)

X_validation_normal = normal_vectorizer.transform(
    normal_validation_texts
)

print("Normal TF-IDF")
print("Train:", X_train_normal.shape)
print("Validation:", X_validation_normal.shape)

Normal TF-IDF
Train: (124, 3753)
Validation: (26, 3753)


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

normal_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    )
)

normal_model.fit(
    X_train_normal,
    Y_train
)

print("Normal text model trained successfully.")

Normal text model trained successfully.


In [62]:
# ADIM 4
# Validation probability predictions

normal_probs = normal_model.predict_proba(
    X_validation_normal
)

print("Probability shape:", normal_probs.shape)

print("\nFirst 5 predictions:")
print(normal_probs[:5])

Probability shape: (26, 4)

First 5 predictions:
[[0.3129118  0.25247119 0.2735608  0.25415641]
 [0.27160911 0.25806841 0.27268182 0.2824937 ]
 [0.33317248 0.25730744 0.25659702 0.24458106]
 [0.3751522  0.24191191 0.24472597 0.22649879]
 [0.39847117 0.23910006 0.27432931 0.23240376]]


In [63]:
# ADIM 5
# Probability -> Binary Labels

threshold = 0.30

normal_pred = (
    normal_probs >= threshold
).astype(int)

print("TRUE LABELS:")
print(Y_validation[:5])

print("\nPREDICTED LABELS:")
print(normal_pred[:5])

TRUE LABELS:
[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]

PREDICTED LABELS:
[[1 0 0 0]
 [0 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [64]:
from sklearn.metrics import f1_score

normal_micro_f1 = f1_score(
    Y_validation,
    normal_pred,
    average="micro",
    zero_division=0
)

normal_macro_f1 = f1_score(
    Y_validation,
    normal_pred,
    average="macro",
    zero_division=0
)

print("==============================================")
print("NORMAL TEXT RESULTS")
print("==============================================")

print("Micro F1:", normal_micro_f1)
print("Macro F1:", normal_macro_f1)

NORMAL TEXT RESULTS
Micro F1: 0.64
Macro F1: 0.6386530136530136


In [65]:
# ADIM 7
# Final TF-IDF - Stopword Removed

final_vectorizer = TfidfVectorizer()

X_train_val_clean = final_vectorizer.fit_transform(
    [
        paper["clean_text"]
        for paper in train_val_papers
    ]
)

X_test_clean = final_vectorizer.transform(
    [
        paper["clean_text"]
        for paper in test_papers
    ]
)

print("FINAL TF-IDF")
print("Train + Validation:", X_train_val_clean.shape)
print("Test:", X_test_clean.shape)

FINAL TF-IDF
Train + Validation: (150, 4145)
Test: (29, 4145)


In [67]:
import numpy as np

Y_train_val = np.vstack([
    Y_train,
    Y_validation
])

print("Y_train:", Y_train.shape)
print("Y_validation:", Y_validation.shape)
print("Y_train_val:", Y_train_val.shape)

Y_train: (124, 4)
Y_validation: (26, 4)
Y_train_val: (150, 4)


In [68]:
# ============================================================
# ADIM 8B — FINAL MODEL TRAINING
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

final_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=2000,
        random_state=42
    )
)

final_model.fit(
    X_train_val_clean,
    Y_train_val
)

print("Final model trained successfully.")

Final model trained successfully.


In [69]:
# ============================================================
# ADIM 9 — TEST PREDICTIONS
# ============================================================

test_probs = final_model.predict_proba(
    X_test_clean
)

print("Test probability shape:", test_probs.shape)

print("\nFirst 5 test probabilities:")
print(test_probs[:5])

Test probability shape: (29, 4)

First 5 test probabilities:
[[0.2948064  0.25985806 0.29008872 0.24326262]
 [0.35754203 0.22269569 0.27417504 0.23495721]
 [0.42106685 0.21508462 0.23283966 0.22616082]
 [0.33834651 0.26285922 0.25346157 0.24466165]
 [0.34478257 0.27236039 0.29158498 0.20602296]]


In [70]:
# ============================================================
# ADIM 10 — TEST LABEL PREDICTIONS
# ============================================================

test_threshold = 0.30

test_pred = (
    test_probs >= test_threshold
).astype(int)

print("Threshold:", test_threshold)

print("\nTRUE LABELS:")
print(Y_test[:5])

print("\nPREDICTED LABELS:")
print(test_pred[:5])

Threshold: 0.3

TRUE LABELS:
[[1 0 0 0]
 [1 0 1 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]

PREDICTED LABELS:
[[0 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [71]:
# ============================================================
# ADIM 11 — FINAL TEST EVALUATION
# ============================================================

from sklearn.metrics import (
    f1_score,
    classification_report
)

test_micro_f1 = f1_score(
    Y_test,
    test_pred,
    average="micro",
    zero_division=0
)

test_macro_f1 = f1_score(
    Y_test,
    test_pred,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print("Micro F1:", test_micro_f1)
print("Macro F1:", test_macro_f1)

print("\nClassification Report:")
print(
    classification_report(
        Y_test,
        test_pred,
        target_names=[
            "NLP",
            "Computer Vision",
            "Machine Learning",
            "Robotics"
        ],
        zero_division=0
    )
)

FINAL TEST RESULTS
Micro F1: 0.75
Macro F1: 0.7239448051948052

Classification Report:
                  precision    recall  f1-score   support

             NLP       1.00      0.75      0.86         8
 Computer Vision       0.86      0.75      0.80         8
Machine Learning       0.67      0.25      0.36         8
        Robotics       0.88      0.88      0.88         8

       micro avg       0.88      0.66      0.75        32
       macro avg       0.85      0.66      0.72        32
    weighted avg       0.85      0.66      0.72        32
     samples avg       0.71      0.67      0.68        32



wrong predict 

In [72]:
# ============================================================
# ADIM 12 — ERROR ANALYSIS
# ============================================================

label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

for i, (paper, true_labels, pred_labels) in enumerate(
    zip(test_papers, Y_test, test_pred)
):

    if not np.array_equal(true_labels, pred_labels):

        true_names = [
            label_names[j]
            for j, value in enumerate(true_labels)
            if value == 1
        ]

        pred_names = [
            label_names[j]
            for j, value in enumerate(pred_labels)
            if value == 1
        ]

        print("\n" + "=" * 70)
        print("TEST PAPER:", i)
        print("TITLE:", paper["title"])
        print("TRUE:", true_names)
        print("PREDICTED:", pred_names)


TEST PAPER: 0
TITLE: Panarchy: ripples of a boundary concept
TRUE: ['NLP']
PREDICTED: []

TEST PAPER: 1
TITLE: Neural document expansion for ad-hoc information retrieval
TRUE: ['NLP', 'Machine Learning']
PREDICTED: ['NLP']

TEST PAPER: 5
TITLE: Thirty Musts for Meaning Banking
TRUE: ['NLP']
PREDICTED: []

TEST PAPER: 8
TITLE: A Deep Reinforcement Learning Approach for Ramp Metering Based on Traffic Video Data
TRUE: ['Computer Vision']
PREDICTED: ['Robotics']

TEST PAPER: 9
TITLE: Flexible deep transfer learning by separate feature embeddings and manifold alignment
TRUE: ['Computer Vision']
PREDICTED: ['Computer Vision', 'Machine Learning']

TEST PAPER: 12
TITLE: DNN Architecture for High Performance Prediction on Natural Videos Loses Submodule's Ability to Learn Discrete-World Dataset
TRUE: ['Computer Vision', 'Machine Learning']
PREDICTED: ['Machine Learning']

TEST PAPER: 13
TITLE: Automatic Dataset Augmentation Using Virtual Human Simulation
TRUE: ['Computer Vision', 'Machine Learn

In [73]:
# ============================================================
# ADIM 13 — MACHINE LEARNING ERROR ANALYSIS
# ============================================================

label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

ml_index = label_names.index("Machine Learning")

for i, (paper, true_labels, pred_labels, probs) in enumerate(
    zip(test_papers, Y_test, test_pred, test_probs)
):

    # Gerçek label'lar arasında Machine Learning var mı?
    if true_labels[ml_index] == 1:

        true_names = [
            label_names[j]
            for j, value in enumerate(true_labels)
            if value == 1
        ]

        pred_names = [
            label_names[j]
            for j, value in enumerate(pred_labels)
            if value == 1
        ]

        print("\n" + "=" * 80)
        print("TEST PAPER:", i)
        print("TITLE:", paper["title"])

        print("\nTRUE LABELS:")
        print(true_names)

        print("\nPREDICTED LABELS:")
        print(pred_names)

        print("\nPROBABILITIES:")

        for j, label in enumerate(label_names):
            print(
                f"{label:20s}: {probs[j]:.4f}"
            )


TEST PAPER: 1
TITLE: Neural document expansion for ad-hoc information retrieval

TRUE LABELS:
['NLP', 'Machine Learning']

PREDICTED LABELS:
['NLP']

PROBABILITIES:
NLP                 : 0.3575
Computer Vision     : 0.2227
Machine Learning    : 0.2742
Robotics            : 0.2350

TEST PAPER: 12
TITLE: DNN Architecture for High Performance Prediction on Natural Videos Loses Submodule's Ability to Learn Discrete-World Dataset

TRUE LABELS:
['Computer Vision', 'Machine Learning']

PREDICTED LABELS:
['Machine Learning']

PROBABILITIES:
NLP                 : 0.2741
Computer Vision     : 0.2614
Machine Learning    : 0.3328
Robotics            : 0.2597

TEST PAPER: 13
TITLE: Automatic Dataset Augmentation Using Virtual Human Simulation

TRUE LABELS:
['Computer Vision', 'Machine Learning']

PREDICTED LABELS:
['Computer Vision']

PROBABILITIES:
NLP                 : 0.2462
Computer Vision     : 0.3614
Machine Learning    : 0.2720
Robotics            : 0.2441

TEST PAPER: 16
TITLE: Detecting B

Validation'da label bazlı threshold analizi

In [74]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]

label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

ml_index = 2

print("=" * 80)
print("MACHINE LEARNING — VALIDATION THRESHOLD ANALYSIS")
print("=" * 80)

for threshold in thresholds:

    val_pred_threshold = (val_probs >= threshold).astype(int)

    ml_precision = precision_score(
        Y_validation[:, ml_index],
        val_pred_threshold[:, ml_index],
        zero_division=0
    )

    ml_recall = recall_score(
        Y_validation[:, ml_index],
        val_pred_threshold[:, ml_index],
        zero_division=0
    )

    ml_f1 = f1_score(
        Y_validation[:, ml_index],
        val_pred_threshold[:, ml_index],
        zero_division=0
    )

    print(
        f"\nThreshold: {threshold:.2f}"
    )

    print(
        f"ML Precision: {ml_precision:.3f}"
    )

    print(
        f"ML Recall:    {ml_recall:.3f}"
    )

    print(
        f"ML F1:        {ml_f1:.3f}"
    )

MACHINE LEARNING — VALIDATION THRESHOLD ANALYSIS

Threshold: 0.20
ML Precision: 0.429
ML Recall:    0.857
ML F1:        0.571

Threshold: 0.25
ML Precision: 0.571
ML Recall:    0.571
ML F1:        0.571

Threshold: 0.30
ML Precision: 0.800
ML Recall:    0.571
ML F1:        0.667

Threshold: 0.35
ML Precision: 1.000
ML Recall:    0.429
ML F1:        0.600

Threshold: 0.40
ML Precision: 1.000
ML Recall:    0.286
ML F1:        0.444


## N-gram TF-IDF

In [77]:
train_texts = [
    paper["clean_text"]
    for paper in train_papers
]

validation_texts = [
    paper["clean_text"]
    for paper in validation_papers
]

print("Train texts:", len(train_texts))
print("Validation texts:", len(validation_texts))

Train texts: 124
Validation texts: 26


In [78]:
from sklearn.feature_extraction.text import TfidfVectorizer

ngram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    lowercase=False
)

X_train_ngram = ngram_vectorizer.fit_transform(
    train_texts
)

X_validation_ngram = ngram_vectorizer.transform(
    validation_texts
)

print("N-gram TF-IDF Train:", X_train_ngram.shape)
print("N-gram TF-IDF Validation:", X_validation_ngram.shape)

N-gram TF-IDF Train: (124, 19114)
N-gram TF-IDF Validation: (26, 19114)


In [79]:
ngram_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=2000
    )
)

ngram_model.fit(
    X_train_ngram,
    Y_train
)

print("N-gram model trained successfully.")

N-gram model trained successfully.


In [80]:
ngram_probs = ngram_model.predict_proba(
    X_validation_ngram
)

print("Probability shape:", ngram_probs.shape)

Probability shape: (26, 4)


In [81]:
ngram_preds = (
    ngram_probs >= 0.30
).astype(int)

In [82]:
from sklearn.metrics import f1_score

ngram_micro_f1 = f1_score(
    Y_validation,
    ngram_preds,
    average="micro",
    zero_division=0
)

ngram_macro_f1 = f1_score(
    Y_validation,
    ngram_preds,
    average="macro",
    zero_division=0
)

ngram_ml_f1 = f1_score(
    Y_validation[:, 2],
    ngram_preds[:, 2],
    zero_division=0
)

print("=" * 60)
print("N-GRAM TF-IDF RESULTS")
print("=" * 60)

print("Micro F1:", ngram_micro_f1)
print("Macro F1:", ngram_macro_f1)
print("Machine Learning F1:", ngram_ml_f1)

N-GRAM TF-IDF RESULTS
Micro F1: 0.5909090909090909
Macro F1: 0.5893939393939394
Machine Learning F1: 0.6


In [83]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

min_df_values = [1, 2, 3]

results_min_df = []

for min_df in min_df_values:

    print("\n" + "=" * 60)
    print("MIN_DF:", min_df)
    print("=" * 60)

    # --------------------------------------------------
    # TF-IDF
    # --------------------------------------------------
    vectorizer_min_df = TfidfVectorizer(
        lowercase=False,
        ngram_range=(1, 1),
        min_df=min_df
    )

    X_train_min_df = vectorizer_min_df.fit_transform(
        train_texts
    )

    X_validation_min_df = vectorizer_min_df.transform(
        validation_texts
    )

    print("Train shape:", X_train_min_df.shape)
    print("Validation shape:", X_validation_min_df.shape)

    # --------------------------------------------------
    # MODEL
    # --------------------------------------------------
    model_min_df = OneVsRestClassifier(
        LogisticRegression(
            C=1.0,
            max_iter=2000
        )
    )

    model_min_df.fit(
        X_train_min_df,
        Y_train
    )

    # --------------------------------------------------
    # PREDICTION
    # --------------------------------------------------
    probabilities = model_min_df.predict_proba(
        X_validation_min_df
    )

    predictions = (
        probabilities >= 0.30
    ).astype(int)

    # --------------------------------------------------
    # OVERALL METRICS
    # --------------------------------------------------
    micro_f1 = f1_score(
        Y_validation,
        predictions,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        Y_validation,
        predictions,
        average="macro",
        zero_division=0
    )

    # --------------------------------------------------
    # MACHINE LEARNING METRIC
    # --------------------------------------------------
    ml_f1 = f1_score(
        Y_validation[:, 2],
        predictions[:, 2],
        zero_division=0
    )

    print("Micro F1:", micro_f1)
    print("Macro F1:", macro_f1)
    print("Machine Learning F1:", ml_f1)

    # Sonuçları kaydet
    results_min_df.append({
        "min_df": min_df,
        "features": X_train_min_df.shape[1],
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "ml_f1": ml_f1
    })


MIN_DF: 1
Train shape: (124, 3729)
Validation shape: (26, 3729)
Micro F1: 0.6666666666666666
Macro F1: 0.668956043956044
Machine Learning F1: 0.6666666666666666

MIN_DF: 2
Train shape: (124, 1676)
Validation shape: (26, 1676)
Micro F1: 0.6666666666666666
Macro F1: 0.6684981684981685
Machine Learning F1: 0.6666666666666666

MIN_DF: 3
Train shape: (124, 1071)
Validation shape: (26, 1071)
Micro F1: 0.6071428571428571
Macro F1: 0.6034798534798534
Machine Learning F1: 0.5714285714285714


In [84]:
import pandas as pd

results_min_df_table = pd.DataFrame(
    results_min_df
)

print("\n" + "=" * 70)
print("MIN_DF EXPERIMENT RESULTS")
print("=" * 70)

display(
    results_min_df_table.sort_values(
        "micro_f1",
        ascending=False
    )
)


MIN_DF EXPERIMENT RESULTS


,min_df,features,micro_f1,macro_f1,ml_f1
0,1,3729,0.666667,0.668956,0.666667
1,2,1676,0.666667,0.668498,0.666667
2,3,1071,0.607143,0.603480,0.571429


In [85]:
print("=" * 80)
print("MACHINE LEARNING — VALIDATION ERROR ANALYSIS")
print("=" * 80)

for i in range(len(validation_papers)):

    true_labels = [
        label_names[j]
        for j in range(len(label_names))
        if Y_validation[i][j] == 1
    ]

    predicted_labels = [
        label_names[j]
        for j in range(len(label_names))
        if ngram_preds[i][j] == 1
    ]

    # Gerçek ML ama ML tahmin edilmemiş
    if (
        Y_validation[i][2] == 1
        and ngram_preds[i][2] == 0
    ):

        print("\n" + "-" * 80)

        print("INDEX:", i)

        print(
            "TITLE:",
            validation_papers[i]["title"]
        )

        print(
            "TRUE:",
            true_labels
        )

        print(
            "PREDICTED:",
            predicted_labels
        )

        print(
            "ML PROBABILITY:",
            round(
                ngram_probs[i][2],
                4
            )
        )

MACHINE LEARNING — VALIDATION ERROR ANALYSIS

--------------------------------------------------------------------------------
INDEX: 8
TITLE: Image to Bengali Caption Generation Using Deep CNN and Bidirectional Gated Recurrent Unit
TRUE: ['Computer Vision', 'Machine Learning']
PREDICTED: ['NLP']
ML PROBABILITY: 0.2762

--------------------------------------------------------------------------------
INDEX: 14
TITLE: Unifying Homophily and Heterophily Network Transformation via Motifs
TRUE: ['Machine Learning']
PREDICTED: []
ML PROBABILITY: 0.2981

--------------------------------------------------------------------------------
INDEX: 16
TITLE: Wheel-Rail Interface Condition Estimation (W-RICE)
TRUE: ['Machine Learning']
PREDICTED: []
ML PROBABILITY: 0.2718

--------------------------------------------------------------------------------
INDEX: 17
TITLE: Hop-Hop Relation-aware Graph Neural Networks
TRUE: ['Machine Learning']
PREDICTED: ['Computer Vision']
ML PROBABILITY: 0.2606


In [86]:
import pandas as pd

ml_error_analysis = []

for i in range(len(validation_papers)):

    if Y_validation[i][2] == 1:

        true_labels = [
            label_names[j]
            for j in range(len(label_names))
            if Y_validation[i][j] == 1
        ]

        predicted_labels = [
            label_names[j]
            for j in range(len(label_names))
            if ngram_preds[i][j] == 1
        ]

        ml_error_analysis.append({
            "index": i,
            "title": validation_papers[i]["title"],
            "true_labels": true_labels,
            "predicted_labels": predicted_labels,
            "ml_probability": ngram_probs[i][2],
            "correct_ml": ngram_preds[i][2] == 1
        })

ml_error_df = pd.DataFrame(ml_error_analysis)

display(
    ml_error_df.sort_values(
        "ml_probability",
        ascending=False
    )
)

,index,title,true_labels,predicted_labels,ml_probability,correct_ml
6,18,Personalized fall detection monitoring system ...,[Machine Learning],[Machine Learning],0.324689,True
3,15,Robustness to Spurious Correlations in Text Cl...,[Machine Learning],[Machine Learning],0.311450,True
0,7,Prediction of Chronic Kidney Disease Using Dee...,"[Computer Vision, Machine Learning]",[Machine Learning],0.300906,True
2,14,Unifying Homophily and Heterophily Network Tra...,[Machine Learning],[],0.298055,False
1,8,Image to Bengali Caption Generation Using Deep...,"[Computer Vision, Machine Learning]",[NLP],0.276194,False
4,16,Wheel-Rail Interface Condition Estimation (W-R...,[Machine Learning],[],0.271756,False
5,17,Hop-Hop Relation-aware Graph Neural Networks,[Machine Learning],[Computer Vision],0.260637,False


Embedding Models

In [87]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model hazır!")

/Users/ayberkpalta/Desktop/llm_end/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17866.56it/s]


Embedding model hazır!


In [88]:
sample_text = train_papers[0]["clean_text"]

sample_embedding = embedding_model.encode(sample_text)

print("Embedding shape:", sample_embedding.shape)
print("İlk 10 değer:")
print(sample_embedding[:10])

Embedding shape: (384,)
İlk 10 değer:
[-0.00146374 -0.072834   -0.04091691 -0.09781124 -0.01098387  0.03122001
  0.04907715 -0.07949817 -0.01992175 -0.13171163]


In [89]:
# TRAIN
X_train_embedding = embedding_model.encode(
    train_texts,
    show_progress_bar=True
)

# VALIDATION
X_validation_embedding = embedding_model.encode(
    validation_texts,
    show_progress_bar=True
)

print("Train embedding shape:", X_train_embedding.shape)
print("Validation embedding shape:", X_validation_embedding.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.34it/s]

Train embedding shape: (124, 384)
Validation embedding shape: (26, 384)


In [90]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

embedding_classifier = OneVsRestClassifier(
    LogisticRegression(
        C=1,
        max_iter=2000,
        random_state=42
    )
)

embedding_classifier.fit(
    X_train_embedding,
    Y_train
)

print("Embedding classifier trained successfully!")

Embedding classifier trained successfully!


In [91]:
Y_val_embedding_proba = embedding_classifier.predict_proba(
    X_validation_embedding
)

print("Probability shape:", Y_val_embedding_proba.shape)

print("\nFirst 5 validation probabilities:")
print(Y_val_embedding_proba[:5])

Probability shape: (26, 4)

First 5 validation probabilities:
[[0.6639936  0.10870439 0.2844195  0.09851676]
 [0.44900137 0.12445983 0.26820529 0.15394644]
 [0.6351737  0.10129002 0.2285685  0.14285089]
 [0.7051591  0.11312302 0.19702335 0.12521684]
 [0.68183315 0.11186343 0.2754869  0.11923372]]


In [92]:
from sklearn.metrics import f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("=" * 60)
print("EMBEDDING — VALIDATION THRESHOLD ANALYSIS")
print("=" * 60)

for threshold in thresholds:

    Y_val_pred = (
        Y_val_embedding_proba >= threshold
    ).astype(int)

    micro_f1 = f1_score(
        Y_validation,
        Y_val_pred,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        Y_validation,
        Y_val_pred,
        average="macro",
        zero_division=0
    )

    predicted_positives = Y_val_pred.sum()

    print(
        f"Threshold: {threshold:.2f} | "
        f"Micro F1: {micro_f1:.3f} | "
        f"Macro F1: {macro_f1:.3f} | "
        f"Predicted Positives: {predicted_positives}"
    )

EMBEDDING — VALIDATION THRESHOLD ANALYSIS
Threshold: 0.20 | Micro F1: 0.635 | Macro F1: 0.649 | Predicted Positives: 57
Threshold: 0.25 | Micro F1: 0.746 | Macro F1: 0.744 | Predicted Positives: 39
Threshold: 0.30 | Micro F1: 0.877 | Macro F1: 0.869 | Predicted Positives: 29
Threshold: 0.35 | Micro F1: 0.836 | Macro F1: 0.819 | Predicted Positives: 27
Threshold: 0.40 | Micro F1: 0.784 | Macro F1: 0.723 | Predicted Positives: 23
Threshold: 0.45 | Micro F1: 0.723 | Macro F1: 0.639 | Predicted Positives: 19
Threshold: 0.50 | Micro F1: 0.636 | Macro F1: 0.559 | Predicted Positives: 16


In [93]:
from sklearn.metrics import classification_report

threshold = 0.30

Y_val_embedding_pred = (
    Y_val_embedding_proba >= threshold
).astype(int)

print("=" * 70)
print("EMBEDDING — VALIDATION CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        Y_validation,
        Y_val_embedding_pred,
        target_names=[
            "NLP",
            "Computer Vision",
            "Machine Learning",
            "Robotics"
        ],
        zero_division=0
    )
)

EMBEDDING — VALIDATION CLASSIFICATION REPORT
                  precision    recall  f1-score   support

             NLP       0.78      1.00      0.88         7
 Computer Vision       1.00      1.00      1.00         7
Machine Learning       0.80      0.57      0.67         7
        Robotics       0.88      1.00      0.93         7

       micro avg       0.86      0.89      0.88        28
       macro avg       0.86      0.89      0.87        28
    weighted avg       0.86      0.89      0.87        28
     samples avg       0.87      0.90      0.88        28



In [94]:
X_train_val_embedding = np.vstack([
    X_train_embedding,
    X_validation_embedding
])

Y_train_val_embedding = np.vstack([
    Y_train,
    Y_validation
])

print("Train + Validation embedding:",
      X_train_val_embedding.shape)

print("Train + Validation labels:",
      Y_train_val_embedding.shape)

Train + Validation embedding: (150, 384)
Train + Validation labels: (150, 4)


In [96]:
test_texts = [
    paper["clean_text"]
    for paper in test_papers
]

print("Test texts:", len(test_texts))
print("First test text:")
print(test_texts[0][:500])

Test texts: 29
First test text:
panarchy ripples of a boundary concept how do social ecological systems change over time in holling and colleagues proposed the concept of panarchy which presented social ecological systems as an interacting set of adaptive cycles each of which is produced by the dynamic tensions between novelty and efficiency at multiple scales initially introduced as a conceptual framework and set of metaphors panarchy has gained the attention of scholars across many disciplines and its ideas continue to inspi


In [97]:
X_test_embedding = embedding_model.encode(
    test_texts,
    show_progress_bar=True
)

print("Test embedding:", X_test_embedding.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]

Test embedding: (29, 384)


In [98]:
X_test_embedding = embedding_model.encode(
    test_texts,
    show_progress_bar=True
)

print("Test embedding shape:", X_test_embedding.shape)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.82it/s]

Test embedding shape: (29, 384)


In [99]:
import numpy as np

X_train_val_embedding = np.vstack([
    X_train_embedding,
    X_validation_embedding
])

Y_train_val_embedding = np.vstack([
    Y_train,
    Y_validation
])

print("Train + Validation embeddings:",
      X_train_val_embedding.shape)

print("Train + Validation labels:",
      Y_train_val_embedding.shape)

Train + Validation embeddings: (150, 384)
Train + Validation labels: (150, 4)


In [100]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

final_embedding_model = OneVsRestClassifier(
    LogisticRegression(
        C=1,
        max_iter=2000,
        random_state=42
    )
)

final_embedding_model.fit(
    X_train_val_embedding,
    Y_train_val_embedding
)

print("Final embedding classifier trained successfully!")

Final embedding classifier trained successfully!


In [101]:
Y_test_embedding_proba = final_embedding_model.predict_proba(
    X_test_embedding
)

print("Test probability shape:", Y_test_embedding_proba.shape)

print("\nFirst 5 test probabilities:")
print(Y_test_embedding_proba[:5])

Test probability shape: (29, 4)

First 5 test probabilities:
[[0.50245976 0.13044743 0.2664665  0.14469384]
 [0.6336442  0.12345184 0.4157509  0.0727279 ]
 [0.7449705  0.05607603 0.26088986 0.10129248]
 [0.7590227  0.12744042 0.2518536  0.06276746]
 [0.6321337  0.14366484 0.3921696  0.06248279]]


# Final Test Predict and F1

In [102]:
from sklearn.metrics import f1_score, classification_report

# Validation'da seçtiğimiz threshold
threshold = 0.30

# Probability → Binary labels
Y_test_embedding_pred = (
    Y_test_embedding_proba >= threshold
).astype(int)

# ------------------------------------------------------------
# F1 SCORES
# ------------------------------------------------------------

micro_f1 = f1_score(
    Y_test,
    Y_test_embedding_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    Y_test,
    Y_test_embedding_pred,
    average="macro",
    zero_division=0
)

print("=" * 70)
print("FINAL EMBEDDING TEST RESULTS")
print("=" * 70)

print(f"Threshold: {threshold}")
print(f"Micro F1: {micro_f1:.3f}")
print(f"Macro F1: {macro_f1:.3f}")

# ------------------------------------------------------------
# CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:")

print(
    classification_report(
        Y_test,
        Y_test_embedding_pred,
        target_names=[
            "NLP",
            "Computer Vision",
            "Machine Learning",
            "Robotics"
        ],
        zero_division=0
    )
)

FINAL EMBEDDING TEST RESULTS
Threshold: 0.3
Micro F1: 0.812
Macro F1: 0.820

Classification Report:
                  precision    recall  f1-score   support

             NLP       1.00      1.00      1.00         8
 Computer Vision       0.78      0.88      0.82         8
Machine Learning       0.55      0.75      0.63         8
        Robotics       0.78      0.88      0.82         8

       micro avg       0.76      0.88      0.81        32
       macro avg       0.78      0.88      0.82        32
    weighted avg       0.78      0.88      0.82        32
     samples avg       0.79      0.88      0.82        32



Error ANALYSIS

In [103]:
label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

print("=" * 90)
print("EMBEDDING — TEST ERROR ANALYSIS")
print("=" * 90)

for i in range(len(test_papers)):

    true_labels = [
        label_names[j]
        for j in range(len(label_names))
        if Y_test[i][j] == 1
    ]

    predicted_labels = [
        label_names[j]
        for j in range(len(label_names))
        if Y_test_embedding_pred[i][j] == 1
    ]

    # Sadece hata yapan paper'ları göster
    if true_labels != predicted_labels:

        print("\n" + "-" * 90)
        print("TEST INDEX:", i)
        print("TITLE:", test_papers[i]["title"])
        print("TRUE LABELS:", true_labels)
        print("PREDICTED LABELS:", predicted_labels)

        print("\nPROBABILITIES:")

        for j, label in enumerate(label_names):
            print(
                f"{label:<20}: "
                f"{Y_test_embedding_proba[i][j]:.4f}"
            )

EMBEDDING — TEST ERROR ANALYSIS

------------------------------------------------------------------------------------------
TEST INDEX: 4
TITLE: A Study of Feature Extraction techniques for Sentiment Analysis
TRUE LABELS: ['NLP']
PREDICTED LABELS: ['NLP', 'Machine Learning']

PROBABILITIES:
NLP                 : 0.6321
Computer Vision     : 0.1437
Machine Learning    : 0.3922
Robotics            : 0.0625

------------------------------------------------------------------------------------------
TEST INDEX: 6
TITLE: Benchmarking BioRelEx for Entity Tagging and Relation Extraction
TRUE LABELS: ['NLP']
PREDICTED LABELS: ['NLP', 'Machine Learning']

PROBABILITIES:
NLP                 : 0.6505
Computer Vision     : 0.1305
Machine Learning    : 0.3590
Robotics            : 0.0786

------------------------------------------------------------------------------------------
TEST INDEX: 8
TITLE: A Deep Reinforcement Learning Approach for Ramp Metering Based on Traffic Video Data
TRUE LABELS: ['Co

In [104]:
import pandas as pd

error_rows = []

for i in range(len(test_papers)):

    true_labels = [
        label_names[j]
        for j in range(len(label_names))
        if Y_test[i][j] == 1
    ]

    predicted_labels = [
        label_names[j]
        for j in range(len(label_names))
        if Y_test_embedding_pred[i][j] == 1
    ]

    if true_labels != predicted_labels:

        error_rows.append({
            "index": i,
            "title": test_papers[i]["title"],
            "true_labels": true_labels,
            "predicted_labels": predicted_labels,
            "NLP_probability": Y_test_embedding_proba[i][0],
            "CV_probability": Y_test_embedding_proba[i][1],
            "ML_probability": Y_test_embedding_proba[i][2],
            "Robotics_probability": Y_test_embedding_proba[i][3]
        })

error_df = pd.DataFrame(error_rows)

print("Total test papers:", len(test_papers))
print("Total errors:", len(error_df))

error_df

Total test papers: 29
Total errors: 10


,index,title,true_labels,predicted_labels,NLP_probability,CV_probability,ML_probability,Robotics_probability
0,4,A Study of Feature Extraction techniques for S...,[NLP],"[NLP, Machine Learning]",0.632134,0.143665,0.392170,0.062483
1,6,Benchmarking BioRelEx for Entity Tagging and R...,[NLP],"[NLP, Machine Learning]",0.650476,0.130531,0.358996,0.078627
2,8,A Deep Reinforcement Learning Approach for Ram...,[Computer Vision],[Robotics],0.074854,0.274943,0.245791,0.510788
3,9,Flexible deep transfer learning by separate fe...,[Computer Vision],"[Computer Vision, Machine Learning]",0.192130,0.575395,0.364432,0.086395
4,13,Automatic Dataset Augmentation Using Virtual H...,"[Computer Vision, Machine Learning]",[Computer Vision],0.122236,0.516964,0.227894,0.269938
5,14,DeepSWIR: A Deep Learning Based Approach for t...,[Computer Vision],"[Computer Vision, Machine Learning]",0.102778,0.504208,0.320510,0.179371
6,15,Image-based reconstruction for the impact prob...,[Computer Vision],"[Computer Vision, Machine Learning]",0.075832,0.609458,0.346389,0.183864
7,19,Blackwell Online Learning for Markov Decision ...,[Machine Learning],[Robotics],0.110726,0.084361,0.256160,0.635222
8,20,A Generative Model for Sampling High-Performan...,[Machine Learning],"[Computer Vision, Machine Learning]",0.160789,0.460229,0.389722,0.108840
9,21,Relation-Shape Convolutional Neural Network fo...,[Robotics],[Computer Vision],0.095336,0.613306,0.167684,0.283377


In [105]:
ml_errors = error_df[
    error_df["true_labels"].apply(
        lambda x: "Machine Learning" in x
    )
    |
    error_df["predicted_labels"].apply(
        lambda x: "Machine Learning" in x
    )
]

print("Machine Learning related errors:", len(ml_errors))

ml_errors

Machine Learning related errors: 8


,index,title,true_labels,predicted_labels,NLP_probability,CV_probability,ML_probability,Robotics_probability
0,4,A Study of Feature Extraction techniques for S...,[NLP],"[NLP, Machine Learning]",0.632134,0.143665,0.392170,0.062483
1,6,Benchmarking BioRelEx for Entity Tagging and R...,[NLP],"[NLP, Machine Learning]",0.650476,0.130531,0.358996,0.078627
3,9,Flexible deep transfer learning by separate fe...,[Computer Vision],"[Computer Vision, Machine Learning]",0.192130,0.575395,0.364432,0.086395
4,13,Automatic Dataset Augmentation Using Virtual H...,"[Computer Vision, Machine Learning]",[Computer Vision],0.122236,0.516964,0.227894,0.269938
5,14,DeepSWIR: A Deep Learning Based Approach for t...,[Computer Vision],"[Computer Vision, Machine Learning]",0.102778,0.504208,0.320510,0.179371
6,15,Image-based reconstruction for the impact prob...,[Computer Vision],"[Computer Vision, Machine Learning]",0.075832,0.609458,0.346389,0.183864
7,19,Blackwell Online Learning for Markov Decision ...,[Machine Learning],[Robotics],0.110726,0.084361,0.256160,0.635222
8,20,A Generative Model for Sampling High-Performan...,[Machine Learning],"[Computer Vision, Machine Learning]",0.160789,0.460229,0.389722,0.108840


find beset

In [106]:
from sklearn.metrics import f1_score

label_names = [
    "NLP",
    "Computer Vision",
    "Machine Learning",
    "Robotics"
]

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("=" * 80)
print("CLASS-SPECIFIC THRESHOLD ANALYSIS")
print("=" * 80)

best_thresholds = {}

for class_idx, label in enumerate(label_names):

    best_f1 = -1
    best_threshold = None

    print(f"\n{'-' * 60}")
    print(label)

    for threshold in thresholds:

        y_true = Y_validation[:, class_idx]

        y_pred = (
            Y_val_embedding_proba[:, class_idx] >= threshold
        ).astype(int)

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        print(
            f"Threshold: {threshold:.2f} | "
            f"F1: {f1:.3f}"
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    best_thresholds[label] = best_threshold

    print(
        f"BEST → Threshold: {best_threshold:.2f} | "
        f"F1: {best_f1:.3f}"
    )

print("\n" + "=" * 80)
print("BEST THRESHOLDS")
print("=" * 80)

for label, threshold in best_thresholds.items():
    print(f"{label}: {threshold:.2f}")

CLASS-SPECIFIC THRESHOLD ANALYSIS

------------------------------------------------------------
NLP
Threshold: 0.20 | F1: 0.636
Threshold: 0.25 | F1: 0.778
Threshold: 0.30 | F1: 0.875
Threshold: 0.35 | F1: 0.875
Threshold: 0.40 | F1: 0.875
Threshold: 0.45 | F1: 0.800
Threshold: 0.50 | F1: 0.714
BEST → Threshold: 0.30 | F1: 0.875

------------------------------------------------------------
Computer Vision
Threshold: 0.20 | F1: 0.636
Threshold: 0.25 | F1: 0.824
Threshold: 0.30 | F1: 1.000
Threshold: 0.35 | F1: 0.923
Threshold: 0.40 | F1: 0.833
Threshold: 0.45 | F1: 0.833
Threshold: 0.50 | F1: 0.600
BEST → Threshold: 0.30 | F1: 1.000

------------------------------------------------------------
Machine Learning
Threshold: 0.20 | F1: 0.500
Threshold: 0.25 | F1: 0.500
Threshold: 0.30 | F1: 0.667
Threshold: 0.35 | F1: 0.545
Threshold: 0.40 | F1: 0.250
Threshold: 0.45 | F1: 0.000
Threshold: 0.50 | F1: 0.000
BEST → Threshold: 0.30 | F1: 0.667

-------------------------------------------------